# testing

> Create images for use in testing the pipeline.

In [ ]:
# | default_exp euclid.testing

### Approach

We want the following sets of test images. To ensure they are otherwise identical to the true data, we will create these by copying real data files and replacing the data.

**1. Basic test of ICL profile and colour measurements**
- *for checking that the code is working correctly*
- based on stacked cutout of an individual cluster
- simple Poisson-sampled Sérsic BCG+ICL model
- flat background with constant Gaussian noise matching real image

**2. Test of uncertainty estimates**
- *for evaluating our ability to correctly estimate uncertainties due to the background and use these when determining profile extent, shape, colour, etc.*
- a) Sky patch
    - based on stacked cutout of sky patch
    - spatially varying background with constant Gaussian noise, tuned to match real image
- b) Cluster
    - based on stacked cutout of an individual cluster
    - independent realisation of background with same statistics as sky patch

**3. Full test of ICL profile and colour measurements**
- *for checking the effect of the real background and source masking*
- based on stacked cutout of an individual cluster
- simple Poisson-sampled Sérsic BCG+ICL model
- added to real stacked image: authentic background and sources

**4. End-to-end test of the processing pipeline** (postponed)
- *for determining quality of the post-processing in preserving ICL*
- based on individual calibrated frames for a set of observations, e.g. the sky patch
- simple Sérsic BCG+ICL models
- added to real dither images, after autodark and continuity corrections: authentic background and sources

In [ ]:
# | export

import astropy.units as u
import galsim
import numpy as np
from astropy.io import fits

from nicl import ezgal
from nicl.euclid.data_access import default_data_path
from nicl.utilities import physical_to_angular

In [ ]:
# | hide
# Additional imports used in the examples
from nicl.euclid.data_access import DataAccess, default_data_path

In [ ]:
# | export

outpath = default_data_path("test_images")

In [ ]:
# | export


def get_bcg_icl_mags(z=0.1):
    bcg_I_absmag = -25.0
    icl_I_absmag = -25.5
    bcg_zf = 3.0
    icl_zf = 1.0
    model = ezgal.model("bc03_ssp_z_0.02_chab.model")
    model.set_cosmology(Om=0.3, Ol=0.7, h=0.70, w=-1)
    distmod = model.get_distance_moduli(z, 1)
    bands = {band: f"Euclid-{band}.ecsv" for band in ["Y", "J", "H", "VIS"]}

    def calc_mags(zf, I_absmag):
        mags = {
            band: model.get_observed_absolute_mags(zf=zf, filters=filt, zs=z)
            for band, filt in bands.items()
        }
        model_I_absmag = model.get_absolute_mags(zf=zf, filters="I", zs=z)
        delta_mag = I_absmag - model_I_absmag
        mags = {band: round(mags[band] + delta_mag + distmod, 2) for band in mags}
        return mags

    mags_bcg = calc_mags(bcg_zf, bcg_I_absmag)
    mags_icl = calc_mags(icl_zf, icl_I_absmag)
    return mags_bcg, mags_icl


def _mag_to_flux(mag, zp):
    return 10 ** (-(mag - zp) / 2.5)

### 1. Basic test of ICL profile and colour measurements

The cluster is based on the fiducial values from Bellhouse et al. (2025), which are based on Kluge et al. (2021). By default, it is placed at lower redshift than Bellhouse et al. (2025). Magnitudes and colours are rough.

In [ ]:
# | export


def create_basic_test_images(z=0.1):
    path = default_data_path("Q1_R1_clusters_test", "MCXCJ1754.6+6803")
    hdul = fits.open(path / "EUC_NIR_W-STK_H-MCXCJ1754.6+6803.fits")
    shape = hdul["SCI"].shape
    zp = 23.9  # zeropoint for mJy
    rms = dict(H=0.023, J=0.023, Y=0.023, VIS=0.023)
    source_noise_level = 0.05  # relative noise added to sources

    pixscale = 0.3 * u.arcsec / u.pix

    bcg_appmags, icl_appmags = get_bcg_icl_mags(z)

    bcg_re = (physical_to_angular(22.6 * u.kpc, z) / pixscale).to_value(u.pix)
    bcg_n = 4.6
    bcg_q = 0.6
    bcg_theta = 30

    bcg = galsim.Sersic(n=bcg_n, half_light_radius=bcg_re, flux=1)
    bcg_shape = galsim.Shear(q=bcg_q, beta=bcg_theta * galsim.degrees)
    bcg = bcg.shear(bcg_shape)

    icl_re = (physical_to_angular(189.5 * u.kpc, z) / pixscale).to_value(u.pix)
    icl_n = 0.76
    icl_q = 0.6
    icl_theta = 60

    icl = galsim.Sersic(n=icl_n, half_light_radius=icl_re, flux=1)
    icl_shape = galsim.Shear(q=icl_q, beta=icl_theta * galsim.degrees)
    icl = icl.shear(icl_shape)

    print(f"bcg_appmags: {bcg_appmags}")
    print(f"icl_appmags: {icl_appmags}")
    print(f"bcg_re: {bcg_re:.2f} pixels")
    print(f"icl_re: {icl_re:.2f} pixels")

    basic_test_outpath = outpath / "basic_test"
    basic_test_outpath.mkdir(parents=True, exist_ok=True)
    image = galsim.ImageF(*shape)

    for band in ["H", "J", "Y", "VIS"]:
        hdul["RMS"].data[:] = rms[band]
        bcg_flux = _mag_to_flux(bcg_appmags[band], zp)
        icl_flux = _mag_to_flux(icl_appmags[band], zp)
        print(f"{band} bcg_flux: {bcg_flux}")
        print(f"{band} icl_flux: {icl_flux}")
        print(f"{band} bcg_flux + icl_flux: {bcg_flux + icl_flux}")
        cluster = bcg * bcg_flux + icl * icl_flux
        cluster = cluster.withGSParams(maximum_fft_size=8192 * 2)
        cluster.drawImage(image)
        hdul["SCI"].data[:] = image.array
        hdul.writeto(
            basic_test_outpath / f"cluster_{band}_no_noise.fits", overwrite=True
        )
        background_noise = np.random.normal(scale=rms[band], size=shape)
        source_noise = np.random.normal(
            scale=np.maximum(0, source_noise_level * image.array), size=shape
        )
        hdul["SCI"].data[:] += source_noise + background_noise
        hdul.writeto(basic_test_outpath / f"cluster_{band}.fits", overwrite=True)

In [ ]:
create_basic_test_images()

bcg_appmags: {'Y': 12.85, 'J': 12.66, 'H': 12.47, 'VIS': 13.59}
icl_appmags: {'Y': 12.4, 'J': 12.23, 'H': 12.05, 'VIS': 13.07}
bcg_re: 40.85 pixels
icl_re: 342.50 pixels
H bcg_flux: 37325.015779571986
H icl_flux: 54954.08738576237
H bcg_flux + icl_flux: 92279.10316533435
J bcg_flux: 31332.857243155824
J icl_flux: 46558.60935229582
J bcg_flux + icl_flux: 77891.46659545164
Y bcg_flux: 26302.679918953814
Y icl_flux: 39810.71705534969
Y bcg_flux + icl_flux: 66113.3969743035
VIS bcg_flux: 13304.5441797809
VIS icl_flux: 21478.30474130529
VIS bcg_flux + icl_flux: 34782.84892108619


### 2. Test of uncertainty estimates

TBD

### 3. Full test of ICL profile and colour measurements

TBD

In [ ]:
# Define the skypatch region in EDFS
skypatch_ra = 58.7 * u.deg
skypatch_dec = -50.1 * u.deg
skypatch_radius = 1.5 * u.deg

# Find observations
release_name = "Q1_R1"
da = DataAccess(release_name=release_name)
sky_patch_obs_ids = da.find_observations_for_target(
    ra=skypatch_ra, dec=skypatch_dec, radius=skypatch_radius, fully_contained=False
)

### 4. End-to-end test of the processing pipeline

TBD
